In [0]:
from pyspark.sql.functions import current_timestamp, lit, input_file_name

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

CHICAGO_RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw_files/Chicago_RawData.csv"
DALLAS_RAW  = f"/Volumes/{CATALOG}/{SCHEMA}/raw_files/Dallas_RawData.csv"

BRONZE = dbutils.widgets.get("bronze")
SILVER = dbutils.widgets.get("silver")
GOLD   = dbutils.widgets.get("gold")

print(f"Using Catalog: {CATALOG}")
print(f"Raw files schema: {SCHEMA}")
print(f"Bronze: {BRONZE}")
print(f"Silver: {SILVER}")
print(f"Gold: {GOLD}")
print(f"Chicago path: {CHICAGO_RAW}")
print(f"Dallas path: {DALLAS_RAW}")

In [0]:
chicago_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(CHICAGO_RAW)
)

chicago_bronze = (
    chicago_df
    .withColumn("_source_file", lit("Chicago_RawData.csv"))
    .withColumn("_load_timestamp", current_timestamp())
    .withColumn("_source_city", lit("Chicago"))
)

print(f"Chicago row count: {chicago_bronze.count()}")
print(f"Chicago column count: {len(chicago_bronze.columns)}")
chicago_bronze.printSchema()

In [0]:
chicago_bronze.write \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{BRONZE}.chicago_inspections")

print("foodlens_bronze.chicago_inspections created successfully")
spark.sql(f"SELECT COUNT(*) as row_count FROM {BRONZE}.chicago_inspections").show()

In [0]:
dallas_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(DALLAS_RAW)
)

dallas_bronze = (
    dallas_df
    .withColumn("_source_file", lit("Dallas_RawData.csv"))
    .withColumn("_load_timestamp", current_timestamp())
    .withColumn("_source_city", lit("Dallas"))
)

print(f"Dallas row count: {dallas_bronze.count()}")
print(f"Dallas column count: {len(dallas_bronze.columns)}")
dallas_bronze.printSchema()

In [0]:
dallas_bronze.write \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .saveAsTable(f"{BRONZE}.dallas_inspections")

print("foodlens_bronze.dallas_inspections created successfully")
spark.sql(f"SELECT COUNT(*) as row_count FROM {BRONZE}.dallas_inspections").show()

In [0]:
#Validation
print("=== Bronze Tables ===")
spark.sql(f"SHOW TABLES IN {BRONZE}").show()

print("\n=== Chicago Bronze Sample ===")
spark.sql(f"""
    SELECT `Inspection ID`, `DBA Name`, `AKA Name`, `Facility Type`, `Risk`, 
           `Inspection Date`, `Inspection Type`, `Results` 
    FROM {BRONZE}.chicago_inspections 
    LIMIT 5
""").show(truncate=False)

print("\n=== Dallas Bronze Sample ===")
spark.sql(f"""
    SELECT `Restaurant Name`, `Inspection Type`, `Inspection Date`, 
           `Inspection Score`, `Street Address`, `Zip Code` 
    FROM {BRONZE}.dallas_inspections 
    LIMIT 5
""").show(truncate=False)

In [0]:
print("=== Chicago Quick Stats ===")
spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT `Inspection ID`) as unique_inspections,
        COUNT(DISTINCT `DBA Name`) as unique_businesses,
        MIN(`Inspection Date`) as earliest_date,
        MAX(`Inspection Date`) as latest_date,
        COUNT(CASE WHEN `Violations` IS NULL THEN 1 END) as null_violations
    FROM {BRONZE}.chicago_inspections
""").show(truncate=False)

print("=== Dallas Quick Stats ===")
spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT `Restaurant Name`) as unique_restaurants,
        MIN(`Inspection Date`) as earliest_date,
        MAX(`Inspection Date`) as latest_date,
        ROUND(AVG(`Inspection Score`), 2) as avg_score,
        COUNT(CASE WHEN `Zip Code` IS NULL THEN 1 END) as null_zip_codes
    FROM {BRONZE}.dallas_inspections
""").show(truncate=False)